# Speech Emotion Recognition (MFCC + CNN / LSTM / CNN-LSTM)
Run the cells top to bottom. Set **Runtime → Change runtime type → T4 GPU** first.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!ls -la "/content/drive/MyDrive/ser"

## 1. Unpack project and data
Edit the file names below to match the `ls` output above (EMO-DB zip shown; add RAVDESS/TESS the same way).

In [ ]:
PROJECT_ZIP = "/content/drive/MyDrive/ser/ser_project.zip"
EMODB_ZIP   = "/content/drive/MyDrive/ser/archive (4).zip"
# RAVDESS_ZIP = "/content/drive/MyDrive/ser/ravdess.zip"
# TESS_ZIP    = "/content/drive/MyDrive/ser/tess.zip"

!rm -rf /content/work && mkdir -p /content/work/data/EMODB
!unzip -q "{PROJECT_ZIP}" -d /content/work
!unzip -q "{EMODB_ZIP}" -d /content/work/data/EMODB
# !mkdir -p /content/work/data/RAVDESS && unzip -q "{RAVDESS_ZIP}" -d /content/work/data/RAVDESS
# !mkdir -p /content/work/data/TESS && unzip -q "{TESS_ZIP}" -d /content/work/data/TESS
!find /content/work/data/EMODB -name "*.wav" | wc -l

In [ ]:
%cd /content/work/ser_project
!pip install -q librosa soundfile
!python tests/test_data.py

## 2. Train the three models
Add `--ravdess ../data/RAVDESS` / `--tess ../data/TESS` to `DATA` if you have those datasets.

In [ ]:
DATA = "--emodb ../data/EMODB"
for m in ["cnn", "lstm", "cnnlstm"]:
    !python -m ser.train {DATA} --model {m} --epochs 80 --augment --cache cache

## 3. Results

In [ ]:
import json
from IPython.display import Image, display
print(f"{'model':10s} {'accuracy':>9s} {'macro-F1':>9s}")
for m in ["cnn", "lstm", "cnnlstm"]:
    r = json.load(open(f"outputs/{m}/metrics.json")); print(f"{m:10s} {r['accuracy']:9.3f} {r['macro_f1']:9.3f}")
for m in ["cnn", "lstm", "cnnlstm"]:
    display(Image(f"outputs/{m}/confusion_matrix.png")); display(Image(f"outputs/{m}/training_curves.png"))

## 4. Save to Drive and make the submission zip

In [ ]:
!cp -r outputs "/content/drive/MyDrive/ser/outputs"
!cd /content/work && zip -qr /content/drive/MyDrive/ser/submission.zip ser_project -x "ser_project/cache/*" "*/__pycache__/*"
print("Saved: MyDrive/ser/submission.zip")

## 5. (Optional) Predict on your own recording

In [ ]:
# upload a .wav to /content first, then:
!python -m ser.predict --ckpt outputs/cnnlstm/model.pt --audio /content/my_voice.wav